# 14x · 附录：实验室该测多少个

> **这一章是附录，不在主线上。** 主线（14a–14f）讲的是把一个分数拆成两个能量，再用这两个能量做事。这一章不用那两个能量——它回答的是一个部署问题：一个只能测几十个突变的实验室，我们的模型对它值多少。
> 想读连贯的故事，按 14a → 14b → 14c → 14d → 14e → 14f 走，这一章随时可以单独看。

## 这一章在整个项目里的位置

```
00–13 ✅ 平台 → [第二阶段 F：一条线，从一个分数到两个能量]
                 ├─ 14a 把稳定性变成能量 ✅
                 ├─ 14b 把测序读数变成两种测量 ✅
                 ├─ 14c 把一个分数拆成两个能量 ✅
                 ├─ 14d 把两个能量变成一张位点地图 ✅
                 ├─ 14e 把地图用到一个真实的药上 ✅
                 └─ 14f 把整条线讲出来 ✅

附录（不在这条线上）
                 └─ 14x 实验室该测多少个  ← 你在这里
```

## 背景：前面所有章节问的问题，公司里不这么问

07–09 和 14a–14c 都在回答同一种问题：**给一个全新的蛋白，一个测量都没有，预测得多准？**

但一个真实的蛋白工程项目不是这样的。它是这样的：

> 我们要改造这个酶。实验室这个月能测 **96 个**突变体。测哪 96 个？测完之后，下个月再测 96 个，又该测哪些？

这一章回答两个很具体的问题：

| 问题 | 为什么重要 |
|---|---|
| **测多少个才值得？** | 测量很贵。如果测 24 个还不如不测（直接用免费的预测），那就别测 |
| **测哪些？** | 同样是 96 个，挑着测能不能比随便测找到更好的突变 |

这是**机器学习引导的定向进化**（machine-learning-guided directed evolution）的核心，也是蛋白工程岗位每天在做的事。

## 这一章和 09 号那个模型的关系

一个少量数据的模型要拟合，总得有个**起点**。文献里的标准做法（Hsu 等 2022）用的是一个免费的、不要任何测量的预测——**ESM-2 的零样本分**。它有两个作用：

- **当标尺**：拟合出来的模型，从第几个测量开始才比它好？
- **当特征**：把它当成额外一列喂给回归，这叫"增强"（augmented）模型，公认最稳

但我们自己有一个这样的预测：**09 号注册上线的模型**。它在这批考试蛋白上考过一次，比 ESM-2 零样本明显好。所以这一章不只重做一遍文献的实验，而是把它当成一个**对照**，问一个属于我们自己的问题：

> **把起点从免费的 ESM-2 换成我们自己的模型，实验室能少测多少个？**

所以下面每张表都有**两条起跑线**：ESM-2 的，和我们的。

## 怎么做实验

拿我们**锁死的考试蛋白**当"全新的项目"——模型从没见过它们。对每一组实验：

```
假装只测了 n 个突变（n = 24 … 384，随机抽，重复 5 次）
         ↓
用这 n 个训练一个小模型
         ↓
在剩下的几千个突变上看排得准不准
```

比的几种做法：

| 做法 | 用到什么 | 代表什么 |
|---|---|---|
| **ESM-2 零样本** | 什么都不测，直接用 ESM-2 的预测 | 文献的免费基准线 |
| **我们的分**（稳定性 / 功能） | 什么都不测，直接用 09 号的模型 | **我们自己的免费基准线** |
| **one-hot + ridge** | 只知道"第几位换成什么" | 不含任何生物学知识的下限 |
| **ESM-2 向量 + ridge** | 语言模型的残基向量 | 标准做法 |
| **ESM-2 向量 + 某个分 + ridge** | 再加上一个免费预测当特征 | Hsu 的"增强"版；每条起跑线各做一个 |

代码在 `qafi_core.lown`，性质写成了测试。

---

## 准备

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"   # PyTorch 和 XGBoost 在同一个进程里：只能单线程（见 07f）
import subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
from scipy import stats
from databricks.sdk import WorkspaceClient
from qafi_platform import lakehouse, registry, tracking
from qafi_core import lown, neural, release
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
CATALOG = "qafi_dev"
EXPERIMENT = f"/Users/{w.current_user.me().user_name}/qafi-variant-effect"
ESM2 = "08e4846e537177426273712802403f7ba8261b6c"
MODELS = ROOT / "data/models"

---

## 第 1 步 · 把考试蛋白当成新项目，并用我们自己的模型给它们打分

### 在干什么
取锁死的考试蛋白和它们的 ESM-2 向量，再**从注册表取回 09 号上线的两个模型**，给同一批突变打上我们自己的分。准备好"假装只测了几个"的实验。

### 为什么用考试蛋白
它们满足"全新项目"的全部条件：模型训练时没见过，连它们的亲戚都没见过（07a 按序列相似度整组留出）。而且每组实验有几千个真实测量，可以反复抽样模拟。

### 一个要说清楚的地方：这是第二次问这批蛋白
09a 已经在这批蛋白上开过一次卷（那一次是"模型有多准"）。这一章问的是另一个问题，但用的是同一批蛋白。所以：

- **没有泄漏**：这些蛋白从来没进过训练，模型的分是干净的
- **但也不是一次全新的考试**：同一批数据回答第二个问题，数字该当成**估计**，不是新考卷。真要发表，这一章应该换一批新发表的 DMS 实验

这一条照 09a 的规矩写下来，不藏着。

### 执行
取回模型、打分，约 1 分钟。

In [ ]:
HEAD = registry.load(registry.full_name("stability"), flavor="pytorch")
TREES = registry.load(registry.full_name("trees"), flavor="xgboost")
TABLES = registry.distributions(registry.full_name("trees"))
parts = ROOT / f"data/esm2-650m/{ESM2}-labelled/embeddings"
if not parts.exists():   # 换了电脑：从平台的表里取
    parts = MODELS / "labelled_embeddings.parquet"
    lakehouse.download(w, f"SELECT uniprot_id, position, embedding FROM {CATALOG}.silver.residue_embedding", parts)
if not (MODELS / "test_variants.parquet").exists():
    lakehouse.download(w, f"SELECT * FROM {CATALOG}.gold.fct_variant_features WHERE split = 'test'", MODELS / "test_variants.parquet")

test = pd.read_parquet(MODELS / "test_variants.parquet")
table, index = neural.embedding_table(parts, test.uniprot_id.unique(), MODELS / "test_embeddings.npy")
test["stability"] = neural.score(HEAD, test, table, index, release.RESIDUE_FEATURES)
test["trees"] = TREES.predict(release.features_frame(test))
test = release.scores(test, TABLES)                     # -> stability_score, function_score
embeddings = np.asarray(np.load(MODELS / "test_embeddings.npy", mmap_mode="r"))
where = index.assign(row=np.arange(len(index))).set_index(["uniprot_id", "position"]).row
test["row"] = where.reindex(pd.MultiIndex.from_arrays([test.uniprot_id, test.uniprot_position])).values
test = (test.dropna(subset=["row"]).astype({"row": int})
            .rename(columns={"uniprot_position": "position", "alt_aa": "mut_aa"}))
test["predicted"] = test.esm_delta                      # 文献的起跑线：一个测量都不要
big = test.groupby("dataset_id").filter(lambda g: len(g) >= 600)
print(f"{big.dataset_id.nunique()} 组实验、{big.uniprot_id.nunique()} 个考试蛋白、{len(big):,} 个突变；上线模型版本 {registry.champions()}")
print(big.groupby("data_kind").dataset_id.nunique().rename("组数").to_string())

### 结果怎么看
128 组实验、92 个蛋白、约 22 万个突变。每组至少 600 个突变，这样抽掉 384 个还能剩下够多的来评判。
实验的种类很不均：**Domainome 109 组**（测蛋白量，接近稳定性）、**其他 MaveDB 16 组**（测功能）、ProteinGym 3 组。

`champions` 打印的是注册表里 `@champion` 指向的版本号，和 09c 给全蛋白组打分用的是同一个模型。

---

## 第 2 步 · 测多少个才值得

### 在干什么
对每组实验，随机抽 n 个当"已测"，训练，在剩下的上面看 Spearman。每个预算重复 5 次。

### 背景：为什么免费的预测是一条水平线
免费的预测不拟合任何东西——它不看你测了多少个。所以它在图上是一条**水平线**，正好当标尺：**拟合出来的模型，从第几个测量开始才值得？**

这一章有**三条**水平线：ESM-2 的零样本分，和我们自己的两个分（稳定性分、功能分）。

### 一个要说清楚的选择
我们的两个分，对不同实验各有擅长：稳定性分更适合 Domainome 这类测蛋白量的实验，功能分更适合测活性的实验。**下面两条线都报，不按实验类型挑**——挑了就是用答案选答案。

### 执行
128 组 × 6 个预算 × 5 次 × 8 条线，约 3 分钟。

In [ ]:
PRIORS = {"zero-shot": "predicted", "our stability score": "stability_score", "our function score": "function_score"}
BUDGETS = (24, 48, 96, 192, 288, 384)
ORDER = ["zero-shot", "our stability score", "our function score", "one-hot + ridge", "ESM-2 + ridge",
         "ESM-2 + zero-shot + ridge", "ESM-2 + our stability score + ridge", "ESM-2 + our function score + ridge"]
curves = [lown.curve(assay.reset_index(drop=True), embeddings, budgets=BUDGETS, draws=5, priors=PRIORS).assign(dataset_id=name)
          for name, assay in big.groupby("dataset_id")]
curve = pd.concat(curves, ignore_index=True)
per_assay = (curve.groupby(["dataset_id", "model", "budget"]).spearman.median().reset_index()
                  .merge(big[["dataset_id", "data_kind"]].drop_duplicates(), on="dataset_id"))
per_assay.pivot_table(index="model", columns="budget", values="spearman").loc[ORDER].round(3)

In [ ]:
flat = per_assay[per_assay.budget == 96].pivot_table(index="dataset_id", columns="model", values="spearman")
for ours in ["our stability score", "our function score"]:
    d = flat[ours] - flat["zero-shot"]
    print(f"{ours:22s} 赢 ESM-2 零样本：{(d > 0).mean():.0%} 的实验，平均高 {d.mean():+.3f}，p = {stats.wilcoxon(d).pvalue:.1e}")

wide = per_assay.pivot_table(index=["dataset_id", "budget"], columns="model", values="spearman").reset_index()
def crossover(start, augmented):
    """每组实验里，拟合模型第一次超过这条起跑线所需要的测量数。"""
    out = []
    for _, g in wide.groupby("dataset_id"):
        g = g.sort_values("budget")
        won = g[g[augmented] > g[start]]
        out.append(int(won.budget.iloc[0]) if len(won) else np.inf)
    return pd.Series(out)
for start, augmented in [("zero-shot", "ESM-2 + zero-shot + ridge"),
                         ("our function score", "ESM-2 + our function score + ridge")]:
    r = crossover(start, augmented)
    share = "，".join(f"≤{b}: {(r <= b).mean():.0%}" for b in BUDGETS)
    print(f"\n从「{start}」起步，拟合模型要测多少个才反超：中位数 {r.median():.0f}（{share}；384 个还反超不了：{(r > 384).mean():.0%}）")

### 结果怎么看

| 测了几个 | 24 | 48 | 96 | 192 | 288 | 384 |
|---|---|---|---|---|---|---|
| ESM-2 零样本（什么都不测） | 0.420 | 0.420 | 0.419 | 0.422 | 0.419 | 0.419 |
| **我们的稳定性分**（什么都不测） | **0.588** | **0.588** | **0.588** | 0.589 | 0.588 | 0.587 |
| **我们的功能分**（什么都不测） | **0.576** | **0.576** | **0.576** | 0.577 | 0.576 | 0.577 |
| one-hot + ridge | 0.175 | 0.272 | 0.396 | 0.506 | 0.555 | 0.580 |
| ESM-2 向量 + ridge | 0.323 | 0.388 | 0.448 | 0.512 | 0.542 | 0.559 |
| ESM-2 + 零样本 + ridge | 0.347 | 0.414 | 0.479 | 0.547 | 0.580 | 0.597 |
| ESM-2 + 我们的稳定性分 + ridge | 0.361 | 0.427 | 0.493 | 0.561 | 0.595 | 0.612 |
| **ESM-2 + 我们的功能分 + ridge** | **0.361** | **0.429** | **0.495** | **0.564** | **0.597** | **0.614** |

四条结论：

**1. 起跑线差得非常远。** 一个测量都不做，ESM-2 零样本 0.420，我们的分 **0.588 / 0.576**。配对看：我们的功能分在 **97%** 的实验里赢 ESM-2 零样本，平均高 **0.156**（p = 2e-22）。这就是 07–09 那几章训练出来的东西，在这里变成了一条高得多的起跑线。

**2. 我们的模型大约值 300 个测量。** 按文献的标准做法（ESM-2 + 零样本 + ridge），要测到 **288 个**才追到 0.580，勉强够上我们**免费**的 0.576。换句话说：**一个新项目拿到我们的分，等于白捡了三块 96 孔板的测量。**

**3. "测多少才值"的答案被整个推后了。** 从 ESM-2 起步，中位 **48 个**测量就能反超它；从我们的分起步，中位要 **288 个**，而且有 **37%** 的实验测满 384 个仍然反超不了。
   **这是个好消息，也是个要小心的地方**：起点越好，自己测的数据越难证明自己——项目早期的钱，应该花在别处。

**4. 当特征也有用，但好处小得多。** 把我们的分当成 ridge 的一列（最后两行），比用 ESM-2 的分在每个预算上都高 0.014–0.017。一行代码的事，白拿，但和起跑线那 0.156 比，小一个数量级。

另外，文献的两条结论在我们的数据上照样成立：**测 24 个还不如不测**（0.347 对 0.420），**one-hot 要约 4 倍的测量才追上语言模型向量**（24 个时 0.175 对 0.323）。

### 面试怎么说
> "Measuring only pays once it beats the prediction you already have for free. With a public zero-shot model as the starting point the crossover is a median of 48 measured variants. With our released model it moves to 288, and in 37% of assays 384 measurements still do not beat it — our score is worth roughly three 96-well plates of wet-lab work on a protein nobody has measured. The same score as a ridge feature adds only 0.015: almost all of the value is in the starting point, not the fitting."

---

## 第 3 步 · 测哪些：主动学习

### 在干什么
模拟一个真实的设计循环：测一批、重新拟合、挑下一批、再测。

### 背景

前一步是**随机**抽 n 个。但真实项目是一轮一轮来的，每一轮可以用已有的结果决定下一轮测什么。这叫**主动学习**（active learning）。

两种挑法：

| 挑法 | 怎么选下一批 | 想法 |
|---|---|---|
| **贪心**（greedy） | 预测最好的那 96 个 | 直接往好的方向挖 |
| **UCB** | 预测值 + 不确定性 | 也去看看模型吃不准的地方，免得钻进死胡同 |

对照组是**随机**挑。这里用三个 ridge（不同正则强度）当一个小集成，它们之间的分歧就是"不确定性"。

**这一步同样跑两条起跑线**：模型用 ESM-2 的分当特征，还是用我们的分当特征。

### 怎么算赢
不是看相关系数——项目不关心你整体排得准不准，只关心**有没有把最好的那几个挖出来**。所以看两个数：

- **找到的最好值**（占整个景观里最好值的比例）
- **找到了几个前 1%**

### 执行
每轮 96 个，4 轮（共 384 个），5 种做法各跑 3 个随机种子，约 5 分钟。

In [ ]:
JOBS = {("random", "function_score"): "随机", ("greedy", "predicted"): "贪心 / ESM-2",
        ("ucb", "predicted"): "UCB / ESM-2", ("greedy", "function_score"): "贪心 / 我们的分",
        ("ucb", "function_score"): "UCB / 我们的分"}
campaigns = [lown.campaign(assay.reset_index(drop=True), embeddings, batch=96, rounds=4,
                           strategy=strategy, prior=prior, seed=seed).assign(dataset_id=name, 做法=label)
             for name, assay in big.groupby("dataset_id")
             for (strategy, prior), label in JOBS.items() for seed in range(3)]
campaign = pd.concat(campaigns, ignore_index=True)
campaign.groupby(["做法", "round"])[["best found", "top 1% found"]].median().round(3)

In [ ]:
last = campaign[campaign["round"] == 4]
print(last.groupby("做法")[["best found", "top 1% found"]].agg(["median", "mean"]).round(3).to_string(), "\n")
found = last.pivot_table(index="dataset_id", columns="做法", values="top 1% found", aggfunc="mean")
for a, b in [("UCB / 我们的分", "随机"), ("UCB / ESM-2", "随机"),
             ("UCB / 我们的分", "UCB / ESM-2"), ("UCB / 我们的分", "贪心 / 我们的分")]:
    d = (found[a] - found[b]).dropna()
    p = stats.wilcoxon(d).pvalue if (d != 0).any() else np.nan
    print(f"{a:16s} 对 {b:12s}：平均多找到 {d.mean():+.2f} 个，{(d > 0).mean():.0%} 的实验更好，p = {p:.1e}")

### 结果怎么看

4 轮（384 个）之后，128 组实验的中位数：

| 做法 | 找到的最好值 | 找到几个前 1%（中位 / 平均） |
|---|---|---|
| 随机 | 0.960 | 4 / 6.40 |
| 贪心 / ESM-2 | 0.998 | 6 / 9.25 |
| UCB / ESM-2 | 0.999 | 7 / 9.30 |
| 贪心 / 我们的分 | **1.000** | 7 / 9.50 |
| **UCB / 我们的分** | **1.000** | **7 / 9.51** |

配对比较（每组实验比一次）：

| 比较 | 平均多找到 | 更好的实验占比 | p |
|---|---|---|---|
| **UCB / 我们的分 对 随机** | **+3.11 个** | 91% | 2.6e-20 |
| UCB / ESM-2 对 随机 | +2.91 个 | 84% | 1.5e-19 |
| UCB / 我们的分 对 UCB / ESM-2 | +0.21 个 | 43% | 1.4e-04 |
| UCB 对 贪心（都用我们的分） | +0.01 个 | 27% | 0.65 |

三条结论：

**1. 让模型挑，比随便挑多找到约一半的好突变。** 7 个对 4 个（平均 9.5 对 6.4），91% 的实验更好。随机挑停在最好值的 0.960，引导挑选基本摸到了全局最优（1.000）。**同样 4 块板的钱，好东西多了近一倍。**

**2. 用哪个模型当起点，在循环里几乎不重要。** 我们的分比 ESM-2 多找到 0.21 个（统计上显著，p = 1e-4，但只有 0.2 个突变）。这和第 2 步的 0.156 形成鲜明对比。

> **为什么？** 第一轮 96 个测完之后，模型手里已经有 96 个这个蛋白自己的真实测量了。到这时候，**数据说的话盖过了起点说的话**。模型的价值几乎全部集中在**一个测量都还没有**的那一刻。

   这一条是这一章最有用的发现，而且它是我们**自己的模型和 ESM-2 比**才看得出来的——只跑文献的对照看不到。

**3. UCB 不比贪心好**（p = 0.65）。文献说加探索有用，但在这种"位置独立"的景观上有限；我们的数据支持"有限"这一边。

**换成项目语言**：买我们的模型，省的是**项目启动时那三块板**；真进了设计循环，省的是**随机挑**那部分（约 75% 更多的优质突变）。两笔账要分开算。

### 面试怎么说
> "I ran the loop a protein engineering project actually runs: 96 per round, four rounds, refit in between. Guided selection finds 7 of the top 1% against 4 for random picking — 91% of assays better, p = 3e-20. But swapping the free prior from a public zero-shot model to our released model adds only 0.2 variants, against a 0.156 Spearman gap when nothing is measured: once 96 real measurements are in, the data outweighs the prior. The model's value is front-loaded, and that is the number that tells a lab where to spend."

---

## 第 4 步 · 记账

In [ ]:
best = per_assay.pivot_table(index="model", columns="budget", values="spearman")
last_round = campaign[campaign["round"] == 4].groupby("做法")[["best found", "top 1% found"]].median()
metrics = {f"spearman_{int(b)}_{m.replace(' ', '_').replace('+', 'and')}": float(best.loc[m, b])
           for m in best.index for b in best.columns}
metrics |= {f"round4_{s}_{k.replace(' ', '_').replace('%', 'pct')}": float(last_round.loc[s, k])
            for s in last_round.index for k in last_round.columns}
metrics |= {"crossover_median_from_zero_shot": float(crossover("zero-shot", "ESM-2 + zero-shot + ridge").median()),
            "crossover_median_from_our_score":
                float(crossover("our function score", "ESM-2 + our function score + ridge").median())}
tracking.experiment_run(
    EXPERIMENT, "low_n_and_design",
    params={"assays": int(big.dataset_id.nunique()), "proteins": int(big.uniprot_id.nunique()),
            "budgets": ",".join(map(str, BUDGETS)), "draws": 5, "batch": 96, "rounds": 4, "seeds": 3,
            "priors": ",".join(PRIORS), "champions": str(registry.champions()),
            "proteins_are": "locked test side, second question asked of them"},
    metrics=metrics, tables={"curve": per_assay, "campaign": campaign}, chapter="14x")
print("已记录"); last_round.round(3)

---

## 这一章做完了什么

- 把锁死的考试蛋白当成 **128 个"全新项目"**，并且**把 09 号上线的模型接了进来**：这一章比的不是文献的对照，而是**我们自己的分当起点能省多少测量**
- **我们的模型值约 300 个测量**：一个测量都不做，我们的分 0.576，而文献的标准做法要测到 **288 个**才追上（0.580）。配对看，97% 的实验里我们的分赢 ESM-2 零样本，平均高 0.156
- **"测多少才值"的门槛被推后了**：从 ESM-2 起步中位 48 个就该开始测；从我们的分起步要 **288 个**，37% 的实验测满 384 个还反超不了
- **模型的价值是前置的**：进了设计循环以后，用我们的分还是用 ESM-2 只差 0.2 个优质突变——第一轮 96 个测完，数据就盖过了起点
- **设计循环本身很值**：引导挑选比随机挑多找到约 75% 的前 1% 突变（7 个对 4 个，91% 的实验更好），并基本摸到全局最优；UCB 不比贪心好

## 这一章为什么在主线之外

主线是 **14a → 14b → 14c → 14d → 14e → 14f**：把一个分数拆成两个能量，再用这两个能量读懂一个蛋白、做一次用药判断。每一章的产出都是下一章的原料。

这一章不用那两个能量，也没有后面的章节用它的产出。它回答的是另一类问题——**部署**：一个真实项目手里只有几十个测量的时候，这个模型值多少钱。所以它放在附录。

要接着读主线，回到 **14d · 把两个能量变成一张位点地图**。

## 提交

In [ ]:
run("git status --short")

In [ ]:
run("git add -A && git commit -q -m 'feat: 14x measure our released model against the free zero-shot start' && git log --oneline -1")